# Home Credit — E02-D robustness check

Notebook này kiểm tra độ bền của **một** feature ứng viên duy nhất,
`CREDIT_GOODS_DIFF`, trên ba validation seed, rồi chốt `E03-BASE`.

Quy tắc quyết định đã được khóa và commit trước khi chạy, tại
`docs/experiments/e02_d_robustness_preregistration.md`. Không sửa quy tắc sau
khi thấy kết quả.

Phạm vi: chỉ hai cấu hình, khác nhau đúng một cột. Không chạy `CREDIT_ANNUITY_RATIO`,
không chạy nhóm recompute, không tuning, không đổi model, không dùng leaderboard.


## Quy tắc quyết định (khóa trước khi chạy)

Hai cấu hình:

- `E01` — locked E01 feature matrix.
- `D` — `E01 + CREDIT_GOODS_DIFF`.

Ba validation seed `42`, `52`, `62`. Trong mỗi seed, hai cấu hình dùng **cùng
một fold list**, xác nhận bằng `fold_fingerprint` trùng nhau. Model random seed
cố định `42`; LightGBM parameters giữ nguyên cấu hình đã khóa từ E01.

`CREDIT_GOODS_DIFF` chỉ được đưa vào baseline nếu đạt **đồng thời** cả bốn:

1. Delta OOF dương ở ít nhất **2/3 seed**.
2. Ít nhất **10/15 fold delta** dương.
3. **Mean** của ba delta OOF theo seed dương.
4. **Trimmed mean** của 15 fold delta dương, sau khi bỏ một giá trị lớn nhất và
   một giá trị nhỏ nhất.

```text
Đạt      ->  E03-BASE = E01 + CREDIT_GOODS_DIFF
Không đạt ->  E03-BASE = E01
```

Nếu ứng viên hoàn toàn không có tác dụng, xác suất qua tiêu chí 2 thuần do ngẫu
nhiên là khoảng 15 phần trăm, và thực tế cao hơn vì các fold không độc lập.
Quy tắc thiên về giữ ứng viên. Kết quả đạt quy tắc **không** được mô tả là đã
chứng minh ưu thế.

Chỉ lượt chạy `baseline` mới có hiệu lực quyết định. Lượt `smoke` dùng dữ liệu
mẫu và số fold khác, nên chỉ để kiểm tra pipeline.


In [ ]:
# 1. Clone public repository and import reusable project code
import gc
import importlib
import platform
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_URL = "https://github.com/ManhTanTran/Qaci-datascience.git"
REPO_BRANCH = "main"
REPO_COMMIT = None  # Pin the pre-registration commit for the final run.
REPO_DIR = Path("/kaggle/working/Qaci-datascience")

def run_git(*arguments: str) -> None:
    subprocess.run(["git", "-C", str(REPO_DIR), *arguments], check=True)

if not (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
if REPO_COMMIT is None:
    run_git("checkout", REPO_BRANCH)
    run_git("pull", "--ff-only", "origin", REPO_BRANCH)
else:
    run_git("fetch", "--depth", "1", "origin", REPO_COMMIT)
    run_git("checkout", "--detach", REPO_COMMIT)

GIT_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
SRC_DIR = (REPO_DIR / "src").resolve()
sys.path.insert(0, str(SRC_DIR))
importlib.invalidate_caches()

from credit_scoring.artifacts import export_dataframe_artifact, export_json_artifact
from credit_scoring.data.home_credit import audit_home_credit_data, load_home_credit_data
from credit_scoring.evaluation.cross_validation import create_stratified_folds
from credit_scoring.experiments.home_credit_credit_amount_factorial import (
    prepare_credit_amount_factorial_data,
)
from credit_scoring.features.home_credit_credit_amount_factorial import (
    describe_credit_amount_factors,
    summarize_feature_matrix_differences,
)
from credit_scoring.modeling.lightgbm_model import run_lightgbm_cv
from credit_scoring.reproducibility import set_global_seed

set_global_seed(42)
PREREGISTRATION_DOC = REPO_DIR / "docs/experiments/e02_d_robustness_preregistration.md"
if not PREREGISTRATION_DOC.is_file():
    raise FileNotFoundError(
        "Pre-registration doc not found in the cloned repository. Commit and push "
        "docs/experiments/e02_d_robustness_preregistration.md before running."
    )
print("Git commit:", GIT_COMMIT)
print("Python:", platform.python_version())
print("Pre-registration present:", PREREGISTRATION_DOC.relative_to(REPO_DIR))

In [ ]:
# 2. Locked configuration — frozen by the pre-registration commit
RUN_MODES = {
    "smoke": {
        "sample_size": 5_000,
        "n_splits": 3,
        "n_estimators": 300,
        "early_stopping_rounds": 50,
    },
    "baseline": {
        "sample_size": None,
        "n_splits": 5,
        "n_estimators": 5_000,
        "early_stopping_rounds": 200,
    },
}
CONFIG = {
    "experiment_name": "E02_D_robustness",
    "run_mode": "smoke",  # Change to baseline only after smoke passes.
    "data_dir": "/kaggle/input/competitions/home-credit-default-risk",
    "output_dir": "/kaggle/working/home_credit_outputs",
    "model_random_state": 42,
    "validation_seeds": [42, 52, 62],
}
if CONFIG["run_mode"] not in RUN_MODES:
    raise ValueError(f"Unknown run mode: {CONFIG['run_mode']}")
MODE = RUN_MODES[CONFIG["run_mode"]]

CANDIDATE_FEATURE = "CREDIT_GOODS_DIFF"
E01_REFERENCE_OOF_AUC = 0.768696
E01_REPLAY_TOLERANCE = 0.0005

# Decision rule; see the pre-registration document. Do not edit after commit.
DECISION_RULE = {
    "min_positive_seeds": 2,
    "min_positive_folds": 10,
    "require_positive_mean_seed_delta": True,
    "require_positive_trimmed_fold_delta": True,
}

MODEL_CONFIG = {
    "learning_rate": 0.02,
    "n_estimators": MODE["n_estimators"],
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 80,
    "subsample": 0.8,
    "colsample_bytree": 0.7,
    "reg_alpha": 0.1,
    "reg_lambda": 5.0,
    "random_state": CONFIG["model_random_state"],
    "n_jobs": -1,
    "verbosity": -1,
}
BASE_VALIDATION_CONFIG = {
    "n_splits": MODE["n_splits"],
    "shuffle": True,
    "early_stopping_rounds": MODE["early_stopping_rounds"],
    "keep_models": False,
}
EXPECTED_FOLD_TOTAL = MODE["n_splits"] * len(CONFIG["validation_seeds"])
IS_DECISIVE_RUN = CONFIG["run_mode"] == "baseline"

OUTPUT_DIR = (
    Path(CONFIG["output_dir"])
    / CONFIG["experiment_name"]
    / CONFIG["run_mode"]
    / GIT_COMMIT[:8]
).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Run mode:", CONFIG["run_mode"], "| decisive:", IS_DECISIVE_RUN)
print("Seeds:", CONFIG["validation_seeds"], "| folds per seed:", MODE["n_splits"])
print("Output directory:", OUTPUT_DIR)

In [ ]:
# 3. Load only application_train and application_test
data = load_home_credit_data(
    CONFIG["data_dir"],
    tables=("application_train", "application_test"),
    nrows=MODE["sample_size"],
    reduce_memory=True,
    validate=True,
)
train = data["application_train"]
test = data["application_test"]
assert train["SK_ID_CURR"].is_unique
assert test["SK_ID_CURR"].is_unique
assert train["TARGET"].isin([0, 1]).all()
display(audit_home_credit_data(data))
print("Rows train/test:", len(train), "/", len(test))
print("Target rate:", float(train["TARGET"].mean()))

In [ ]:
# 4. Build both feature matrices once; only the folds change across seeds
# Both configurations go through the same tested factorial builder, so the
# candidate column is identical to the one used in the factorial ablation.
e01_data = prepare_credit_amount_factorial_data(train, test, factors=())
d_data = prepare_credit_amount_factorial_data(train, test, factors=("D",))

target = e01_data.target
target_array = target.to_numpy()
FEATURE_SETS = {
    "E01": {"train": e01_data.train_features, "test": e01_data.test_features},
    "D": {"train": d_data.train_features, "test": d_data.test_features},
}

spec = describe_credit_amount_factors(("D",))
assert spec.added_columns == (CANDIDATE_FEATURE,)
assert spec.overwritten_columns == (), "the D factor must not overwrite any E01 column"
assert e01_data.categorical_features == d_data.categorical_features
assert np.array_equal(e01_data.train_ids.to_numpy(), d_data.train_ids.to_numpy())

# Raises if the candidate silently changed any E01 column, in value or dtype.
difference_report = summarize_feature_matrix_differences(
    e01_data.train_features,
    d_data.train_features,
    explicitly_overwritten=(),
)
assert list(difference_report["feature"]) == [CANDIDATE_FEATURE]
assert d_data.train_features[CANDIDATE_FEATURE].dtype == np.float32

print("E01 features:", e01_data.train_features.shape[1])
print("D features:", d_data.train_features.shape[1])
print(
    "Candidate missing rate:",
    float(d_data.train_features[CANDIDATE_FEATURE].isna().mean()),
)
display(difference_report)
display(d_data.train_features[CANDIDATE_FEATURE].describe())

In [ ]:
# 5. Run both configurations on every seed, sharing one fold list per seed
seed_rows = []
fold_rows = []
artifact_index = {}
fingerprints = {}

for seed in CONFIG["validation_seeds"]:
    folds = create_stratified_folds(
        target_array,
        n_splits=MODE["n_splits"],
        shuffle=True,
        random_state=seed,
    )
    fold_assignment = np.full(len(target_array), -1, dtype=np.int8)
    for fold_number, (_, valid_index) in enumerate(folds, start=1):
        fold_assignment[np.asarray(valid_index, dtype=int)] = fold_number
    assert np.all(fold_assignment > 0)
    assert len(np.unique(fold_assignment)) == MODE["n_splits"]

    seed_results = {}
    seed_fingerprint = None
    for config_name, matrices in FEATURE_SETS.items():
        result = run_lightgbm_cv(
            matrices["train"],
            target_array,
            matrices["test"],
            categorical_features=e01_data.categorical_features,
            model_config=MODEL_CONFIG,
            validation_config={**BASE_VALIDATION_CONFIG, "random_state": seed},
            folds=folds,
        )
        current_fingerprint = str(result["metadata"]["fold_fingerprint"])
        if seed_fingerprint is None:
            seed_fingerprint = current_fingerprint
        # Central claim of this experiment: both arms saw identical folds.
        assert current_fingerprint == seed_fingerprint, (
            f"seed {seed}: {config_name} ran on a different fold assignment"
        )
        assert np.all(result["validation_counts"] == 1)
        assert np.isfinite(result["oof_predictions"]).all()
        assert np.isfinite(result["test_predictions"]).all()

        if IS_DECISIVE_RUN and seed == 42 and config_name == "E01":
            gap = abs(float(result["oof_auc"]) - E01_REFERENCE_OOF_AUC)
            if gap > E01_REPLAY_TOLERANCE:
                raise RuntimeError(
                    f"E01 seed 42 replay is off by {gap:.6f}, above the locked "
                    f"tolerance {E01_REPLAY_TOLERANCE}. Stop and investigate; "
                    "do not treat this as an environment difference."
                )
            print(f"E01 replay gap vs locked reference: {gap:.6f} (within tolerance)")

        experiment_dir = OUTPUT_DIR / "experiments" / f"seed_{seed}" / config_name
        artifact_index[f"seed_{seed}/{config_name}"] = {
            "oof_predictions": str(export_dataframe_artifact(pd.DataFrame({
                "SK_ID_CURR": e01_data.train_ids.to_numpy(),
                "TARGET": target_array,
                "FOLD": fold_assignment,
                "OOF_PREDICTION": result["oof_predictions"],
                "VALIDATION_COUNT": result["validation_counts"],
            }), experiment_dir / "oof_predictions.csv")),
            "test_predictions": str(export_dataframe_artifact(pd.DataFrame({
                "SK_ID_CURR": e01_data.test_ids.to_numpy(),
                "TEST_PREDICTION": result["test_predictions"],
            }), experiment_dir / "test_predictions.csv")),
            "feature_importance": str(export_dataframe_artifact(
                result["feature_importance"], experiment_dir / "feature_importance.csv"
            )),
        }
        seed_results[config_name] = {
            "oof_auc": float(result["oof_auc"]),
            "fold_scores": np.asarray(result["fold_scores"], dtype=float),
            "mean_auc": float(result["mean_auc"]),
            "std_auc": float(result["std_auc"]),
            "runtime": float(result["runtime"]),
            "n_features": int(matrices["train"].shape[1]),
        }
        result["fitted_models"].clear()
        del result
        gc.collect()

    fingerprints[seed] = seed_fingerprint
    export_dataframe_artifact(
        pd.DataFrame({
            "SK_ID_CURR": e01_data.train_ids.to_numpy(),
            "FOLD": fold_assignment,
        }),
        OUTPUT_DIR / "experiments" / f"seed_{seed}" / "fold_assignments.csv",
    )

    delta_oof = seed_results["D"]["oof_auc"] - seed_results["E01"]["oof_auc"]
    fold_deltas = seed_results["D"]["fold_scores"] - seed_results["E01"]["fold_scores"]
    seed_rows.append({
        "seed": seed,
        "fold_fingerprint": seed_fingerprint,
        "n_features_e01": seed_results["E01"]["n_features"],
        "n_features_d": seed_results["D"]["n_features"],
        "oof_auc_e01": seed_results["E01"]["oof_auc"],
        "oof_auc_d": seed_results["D"]["oof_auc"],
        "delta_oof_auc": float(delta_oof),
        "mean_fold_auc_e01": seed_results["E01"]["mean_auc"],
        "mean_fold_auc_d": seed_results["D"]["mean_auc"],
        "std_fold_auc_e01": seed_results["E01"]["std_auc"],
        "std_fold_auc_d": seed_results["D"]["std_auc"],
        "positive_fold_count": int((fold_deltas > 0).sum()),
        "runtime_seconds_e01": seed_results["E01"]["runtime"],
        "runtime_seconds_d": seed_results["D"]["runtime"],
    })
    for fold_number, delta in enumerate(fold_deltas, start=1):
        fold_rows.append({
            "seed": seed,
            "fold": fold_number,
            "auc_e01": float(seed_results["E01"]["fold_scores"][fold_number - 1]),
            "auc_d": float(seed_results["D"]["fold_scores"][fold_number - 1]),
            "delta_auc": float(delta),
        })
    print(
        f"seed {seed}: E01={seed_results['E01']['oof_auc']:.6f} "
        f"D={seed_results['D']['oof_auc']:.6f} "
        f"delta={delta_oof:+.6f} "
        f"positive folds={int((fold_deltas > 0).sum())}/{MODE['n_splits']}"
    )

seed_summary = pd.DataFrame(seed_rows)
fold_metrics = pd.DataFrame(fold_rows)
assert len(fold_metrics) == EXPECTED_FOLD_TOTAL
assert seed_summary["fold_fingerprint"].nunique() == len(CONFIG["validation_seeds"]), (
    "Different seeds produced identical fold assignments"
)
display(seed_summary)

In [ ]:
# 6. Apply the pre-registered decision rule; no reinterpretation
seed_deltas = seed_summary["delta_oof_auc"].to_numpy(dtype=float)
all_fold_deltas = fold_metrics["delta_auc"].to_numpy(dtype=float)
sorted_fold_deltas = np.sort(all_fold_deltas)
trimmed_fold_deltas = sorted_fold_deltas[1:-1]

positive_seeds = int((seed_deltas > 0).sum())
positive_folds = int((all_fold_deltas > 0).sum())
mean_seed_delta = float(seed_deltas.mean())
trimmed_mean_fold_delta = float(trimmed_fold_deltas.mean())

criteria = {
    "1_positive_seeds": {
        "observed": positive_seeds,
        "threshold": f">= {DECISION_RULE['min_positive_seeds']} of {len(seed_deltas)}",
        "passed": positive_seeds >= DECISION_RULE["min_positive_seeds"],
    },
    "2_positive_folds": {
        "observed": positive_folds,
        "threshold": f">= {DECISION_RULE['min_positive_folds']} of {len(all_fold_deltas)}",
        "passed": positive_folds >= DECISION_RULE["min_positive_folds"],
    },
    "3_mean_seed_delta": {
        "observed": mean_seed_delta,
        "threshold": "> 0",
        "passed": mean_seed_delta > 0,
    },
    "4_trimmed_mean_fold_delta": {
        "observed": trimmed_mean_fold_delta,
        "threshold": "> 0 after dropping one max and one min",
        "passed": trimmed_mean_fold_delta > 0,
    },
}
candidate_accepted = all(item["passed"] for item in criteria.values())
e03_base = ["E01", CANDIDATE_FEATURE] if candidate_accepted else ["E01"]

decision = {
    "run_mode": CONFIG["run_mode"],
    "is_decisive_run": IS_DECISIVE_RUN,
    "candidate_feature": CANDIDATE_FEATURE,
    "criteria": criteria,
    "candidate_accepted": bool(candidate_accepted),
    "e03_base": e03_base,
    "mean_seed_delta": mean_seed_delta,
    "trimmed_mean_fold_delta": trimmed_mean_fold_delta,
    "min_fold_delta": float(sorted_fold_deltas[0]),
    "max_fold_delta": float(sorted_fold_deltas[-1]),
}

display(pd.DataFrame(criteria).T)
print(f"\nCandidate accepted: {candidate_accepted}")
print("E03-BASE =", " + ".join(e03_base))
if not IS_DECISIVE_RUN:
    print(
        "\nSMOKE RUN — this verdict is not binding. It only checks that the "
        "pipeline runs and the rule evaluates. Re-run with run_mode='baseline'."
    )

In [ ]:
# 7. Export artifacts and reproducibility metadata
def installed_version(package_name: str) -> str | None:
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None

robustness_rows = fold_metrics.copy()
robustness_rows["scope"] = "fold"
seed_scope = seed_summary[["seed", "oof_auc_e01", "oof_auc_d", "delta_oof_auc"]].copy()
seed_scope["scope"] = "seed"
robustness_summary = pd.concat([robustness_rows, seed_scope], ignore_index=True)

summary_path = export_dataframe_artifact(seed_summary, OUTPUT_DIR / "seed_summary.csv")
fold_path = export_dataframe_artifact(fold_metrics, OUTPUT_DIR / "fold_metrics.csv")
robustness_path = export_dataframe_artifact(
    robustness_summary, OUTPUT_DIR / "robustness_summary.csv"
)
config_path = export_json_artifact({
    "config": CONFIG,
    "mode": MODE,
    "model_config": MODEL_CONFIG,
    "validation_config": BASE_VALIDATION_CONFIG,
    "decision_rule": DECISION_RULE,
    "candidate_feature": CANDIDATE_FEATURE,
    "e01_reference_oof_auc": E01_REFERENCE_OOF_AUC,
    "e01_replay_tolerance": E01_REPLAY_TOLERANCE,
}, OUTPUT_DIR / "config.json")
decision_path = export_json_artifact(decision, OUTPUT_DIR / "decision.json")
metadata_path = export_json_artifact({
    "status": "completed",
    "git_commit": GIT_COMMIT,
    "preregistration_doc": str(PREREGISTRATION_DOC.relative_to(REPO_DIR)),
    "dataset_path": str(Path(CONFIG["data_dir"]).resolve()),
    "n_train": len(train),
    "n_test": len(test),
    "validation_seeds": CONFIG["validation_seeds"],
    "fold_fingerprints": {str(seed): value for seed, value in fingerprints.items()},
    "environment": {
        "python": platform.python_version(),
        "packages": {name: installed_version(name) for name in [
            "numpy", "pandas", "scikit-learn", "lightgbm"
        ]},
    },
    "summary_path": str(summary_path),
    "fold_metrics_path": str(fold_path),
    "robustness_summary_path": str(robustness_path),
    "config_path": str(config_path),
    "decision_path": str(decision_path),
    "experiment_artifacts": artifact_index,
}, OUTPUT_DIR / "run_metadata.json")
print("Output directory:", OUTPUT_DIR)
print("Run metadata:", metadata_path)
print("Decision:", decision_path)

In [ ]:
# 8. Plot measured fold deltas; generated only from actual run results
figure, axis = plt.subplots(figsize=(9, 5))
positions = np.arange(len(fold_metrics))
colors = ["#15803d" if value > 0 else "#b91c1c" for value in fold_metrics["delta_auc"]]
axis.bar(positions, fold_metrics["delta_auc"], color=colors)
axis.axhline(0, color="black", linewidth=1)
axis.axhline(
    trimmed_mean_fold_delta,
    color="#1d4ed8",
    linestyle="--",
    linewidth=1,
    label=f"trimmed mean = {trimmed_mean_fold_delta:+.6f}",
)
axis.set_xticks(positions)
axis.set_xticklabels(
    [f"s{row.seed}f{row.fold}" for row in fold_metrics.itertuples()],
    rotation=45,
    ha="right",
)
axis.set_title(f"{CANDIDATE_FEATURE} — per-fold Δ AUC vs E01")
axis.set_ylabel("Δ fold AUC")
axis.legend()
plt.tight_layout()
plt.show()

display(fold_metrics)

## Sau khi chạy

Lượt `smoke` chỉ để kiểm tra pipeline. Chỉ lượt `baseline` mới có hiệu lực
quyết định.

Sau full baseline run:

1. Tải toàn bộ thư mục output; kiểm tra `decision.json` và `run_metadata.json`.
2. Xác nhận `fold_fingerprint` trùng nhau giữa E01 và D trong cùng seed, và
   khác nhau giữa ba seed.
3. Cập nhật `docs/experiments/experiment_log.md` bằng số lấy từ artifact, sáu
   chữ số thập phân.
4. Chốt `E03-BASE` theo `decision.json`. Nếu ứng viên đạt, đưa
   `CREDIT_GOODS_DIFF` vào `src/` như một family cộng thêm không đụng cột E01
   nào, kèm test khẳng định điều đó.

Không ghi smoke metric như kết quả competition. Không dùng leaderboard để đổi
kết luận. Nếu kết quả đạt quy tắc, ghi rõ ứng viên được chọn vì là cấu hình đơn
giản nhất không có bằng chứng gây hại, không phải vì đã chứng minh ưu thế.
